# 05 字符串与文本处理

pandas 的文本能力都在 `.str` 访问器里：**先选出文本列，再 `.str.方法()`**，整列批量执行，
不用写 for 循环。这一章是后面处理非结构化数据（第 09、10、12 节）的地基。

本节要掌握：

1. `.str` 的常用方法（去空格、截取、判断、替换、拆分、拼接）；
2. 正则表达式在 pandas 里的三种用法：`contains` / `replace` / `extract`；
3. 从自由文本里抽取结构化字段（订单号、金额、手机号、时间、日志级别）；
4. 中文文本归一化（全角、零宽字符、空白）；
5. 简单词频统计。

> 环境：`llm_sft`，从 `python/pandas` 目录启动。

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")

remarks = pd.Series([
    "订单 A1001 已发货，联系电话 138-1234-5678，金额 899.00 元",
    "客户要求：发票抬头「北京晨光科技有限公司」",
    "加急！！！请于 2024-03-20 前送达，联系人 王女士 13912345678",
    "退货原因：尺寸不合适，申请退款 ¥129.9",
    "  多余空格   和\n换行  需要清理  ",
    None,
], name="remark")
remarks

0      订单 A1001 已发货，联系电话 138-1234-5678，金额 899.00 元
1                     客户要求：发票抬头「北京晨光科技有限公司」
2    加急！！！请于 2024-03-20 前送达，联系人 王女士 13912345678
3                          退货原因：尺寸不合适，申请退款 ¥129.9
4                                多余空格   和\n换行  需要清理  
5                                                            NaN
Name: remark, dtype: str

## 1. `.str` 的基本规则

- 只能用在文本列上（pandas 3.0 中默认文本 dtype 是 `str`）；数值列要先 `.astype(str)`；
- 返回的仍然是 Series，索引不变，所以可以直接赋值回 DataFrame；
- **缺失值怎么处理，要看方法类型**：变换类（`upper`/`len`/`replace`/`extract`）保持缺失不变；
  判断类（`contains`/`startswith`/`endswith`）在 `str` dtype 下默认把缺失当作 `False`，
  可以用 `na=` 改；如果列是 `object` 类型，则返回 `NaN`。

In [2]:
text = pd.Series(["  Hello World  ", "Pandas", "数据分析", "ABC-123"])
print("长度      :", text.str.len().tolist())
print("小写      :", text.str.lower().tolist())
print("去空格    :", text.str.strip().tolist())
print("左去空格  :", text.str.lstrip().tolist())
print("前缀判断  :", text.str.startswith("  ").tolist())
print("包含判断  :", text.str.contains("data|数据", case=False, na=False, regex=True).tolist())
print("出现次数  :", text.str.count("a").tolist())
print("切片      :", text.str.slice(0, 4).tolist())

长度      : [15, 6, 4, 7]
小写      : ['  hello world  ', 'pandas', '数据分析', 'abc-123']
去空格    : ['Hello World', 'Pandas', '数据分析', 'ABC-123']
左去空格  : ['Hello World  ', 'Pandas', '数据分析', 'ABC-123']
前缀判断  : [True, False, False, False]
包含判断  : [False, False, True, False]
出现次数  : [0, 2, 0, 0]
切片      : ['  He', 'Pand', '数据分析', 'ABC-']


In [3]:
has_missing = pd.Series(["abc", None, "abd"])

# 1) 变换类：缺失保持缺失
print("len          :", has_missing.str.len().tolist())
print("upper        :", has_missing.str.upper().tolist())
print("replace      :", has_missing.str.replace("a", "A", regex=False).tolist())

# 2) 判断类：str dtype 下缺失默认当 False
print("contains 默认:", has_missing.str.contains("ab").tolist())
print("contains na=True:", has_missing.str.contains("ab", na=True).tolist())

# 3) 老式 object 列：缺失会变成 NaN，筛选时会出问题
print("object 列    :", has_missing.astype(object).str.contains("ab").tolist())
print("→ 所以筛选前统一写 na=False 最省心")

len          : [3.0, nan, 3.0]
upper        : ['ABC', nan, 'ABD']
replace      : ['Abc', nan, 'Abd']
contains 默认: [True, False, True]
contains na=True: [True, True, True]
object 列    : [True, nan, True]
→ 所以筛选前统一写 na=False 最省心


In [4]:
# 数值列没有 .str，必须先转成字符串
nums = pd.Series([100, 200, 300])
try:
    nums.str.len()
except AttributeError as e:
    print("数值列不能直接用 .str →", str(e)[:60])

print("先 astype(str):", nums.astype(str).str.zfill(5).tolist())   # 00100 这种补零

数值列不能直接用 .str → Can only use .str accessor with string values, not integer
先 astype(str): ['00100', '00200', '00300']


## 2. 正则速查

| 语法 | 含义 | 例子 |
| --- | --- | --- |
| `.` | 任意一个字符 | `a.c` 匹配 `abc` |
| `\d` `\w` `\s` | 数字 / 字母数字下划线 / 空白 | `\d{4}` 匹配四位数字 |
| `[abc]` `[^abc]` | 字符集 / 排除 | `[0-9]`、`[^\d]` |
| `*` `+` `?` | 0+ / 1+ / 0-1 次 | `\d+` 匹配一串数字 |
| `{n,m}` | n 到 m 次 | `\d{3,4}` |
| `^` `$` | 行首 / 行尾 | `^\d+$` 整行都是数字 |
| `(...)` | 捕获组 | `(\d{3})-(\d{4})` |
| `(?P<名>...)` | 命名捕获组 | `.str.extract` 里直接变成列名 |
| `a|b` | 或 | `cat|dog` |
| `.*?` | 非贪婪匹配 | `<.*?>` 只匹配到第一个 `>` |

> 在 Python 字符串里写正则，建议用**原始字符串** `r"\d+"`，否则 `\d` 会被当成转义字符。

## 3. 替换：字面量 vs 正则

**重要**：`str.replace()` 的 `regex` 默认是 `False`，要按正则替换必须显式写 `regex=True`。
这是 pandas 2.0 起的变化，也是新手最常踩的坑。

In [5]:
s = pd.Series(["a1b2c3", "138-1234-5678"])

print("字面量替换（默认）:", s.str.replace("1", "#").tolist())      # 只替换字符 1
print("正则替换          :", s.str.replace(r"\d", "#", regex=True).tolist())  # 所有数字

字面量替换（默认）: ['a#b2c3', '#38-#234-5678']
正则替换          : ['a#b#c#', '###-####-####']


In [6]:
# 实战：清洗一段带标点、URL、多余空白的文本
dirty = pd.Series([
    "  【公告】系统升级通知！详情见 https://example.com/notice?id=3   ",
    "客服电话：400-800-1234，邮箱 service@example.com。",
])
step1 = dirty.str.strip()
step2 = step1.str.replace(r"https?://\S+", "[URL]", regex=True)         # 去链接
step3 = step2.str.replace(r"[，。！？；：、]", " ", regex=True)          # 中文标点 → 空格
step4 = step3.str.replace(r"\s+", " ", regex=True)                    # 连续空白 → 一个空格
print(step4.tolist())

['【公告】系统升级通知 详情见 [URL]', '客服电话 400-800-1234 邮箱 service@example.com ']


## 4. 抽取：`extract` 把文本变成几个新列

`extract` 只认**捕获组**：正则里有几个组，就返回几列；用命名组可以直接指定列名。
匹配不到的行返回 `NaN`，不会报错。

In [7]:
extracted = remarks.str.extract(r"订单\s*(?P<order_no>[A-Z]\d+)")
extracted

,order_no
0,A1001
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN


In [8]:
# 一次抽多个字段：分成几个正则写，再横向拼起来（比一个超长正则更好维护）
pd.concat(
    [
        remarks.str.extract(r"订单\s*(?P<order_no>[A-Z]\d+)"),
        remarks.str.extract(r"(?P<amount>\d+\.\d+)\s*元"),
        remarks.str.extract(r"(?P<date>\d{4}-\d{2}-\d{2})"),
    ],
    axis=1,
)

,order_no,amount,date
0,A1001,899.00,NaN
1,NaN,NaN,NaN
2,NaN,NaN,2024-03-20
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN


In [9]:
# extractall：一行里可能有多个匹配，返回 DataFrame（带 match 层级索引）
reasons = pd.Series(["退款 ¥129.9，运费 ¥10.0", "仅退款 ¥59"])
reasons.str.extractall(r"¥(?P<price>\d+\.?\d*)")

price
  match       
0 0      129.9
  1       10.0
1 0         59

## 5. 拆分、拼接、对齐

- `split(sep, expand=True)`：把一列拆成多列，最常用；
- `split(...).explode()`：把一列拆成多行（做词频、标签统计时必用）；
- `cat(sep=)`：把多列拼成一列；
- `pad` / `zfill`：补齐宽度 / 补零；
- `get(i)`：取字符串的第 i 个字符。

In [10]:
path = pd.Series(["/docs/intro.html", "/blog/post-1.html", "/home"])
split_df = path.str.strip("/").str.split("/", expand=True)
split_df.columns = ["一级目录", "文件名"]
split_df

,一级目录,文件名
0,docs,intro.html
1,blog,post-1.html
2,home,NaN


In [11]:
# 一行拆成多行
tags = pd.Series(["数码,家电,配件", "食品", "服饰,配件"])
tags.str.split(",").explode()

0    数码
0    家电
0    配件
1    食品
2    服饰
2    配件
dtype: str

In [12]:
# 多列拼一列；一列拼成一个字符串
people = pd.DataFrame({"姓": ["张", "李"], "名": ["三", "四"]})
people["姓名"] = people["姓"].str.cat(people["名"])
print(people["姓名"].tolist())
print("全部拼起来:", people["姓名"].str.cat(sep="、"))

print("补零到 6 位:", pd.Series(["7", "42", "138"]).str.zfill(6).tolist())
print("右侧补空格到 6 位:", pd.Series(["7", "42"]).str.pad(6, side="right").tolist())

['张三', '李四']
全部拼起来: 张三、李四
补零到 6 位: ['000007', '000042', '000138']
右侧补空格到 6 位: ['7     ', '42    ']


## 6. 实战：把日志行拆成结构化表

非结构化数据最常见的样子就是日志。思路永远是：**一行一条记录 → 正则命名分组 → 变成列**。

In [13]:
logs = pd.Series([
    "2024-06-01 08:12:33 INFO  user=u001 path=/home status=200",
    "2024-06-01 08:13:02 ERROR user=u002 path=/pay status=500",
    "2024-06-01 08:15:44 WARN  user=u001 path=/cart status=302",
])

log_df = logs.str.extract(
    r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+(?P<level>[A-Z]+)\s+"
    r"user=(?P<user>\w+)\s+path=(?P<path>\S+)\s+status=(?P<status>\d+)"
)
log_df["status"] = pd.to_numeric(log_df["status"])
log_df["ts"] = pd.to_datetime(log_df["ts"])
log_df

,ts,level,user,path,status
0,2024-06-01 08:12:33,INFO,u001,/home,200
1,2024-06-01 08:13:02,ERROR,u002,/pay,500
2,2024-06-01 08:15:44,WARN,u001,/cart,302


In [14]:
# 有了结构化字段，就能正常做统计了
log_df.groupby("level").agg(次数=("ts", "count"), 平均状态码=("status", "mean")).round(1)

,次数,平均状态码
level,,
ERROR,1,500.0
INFO,1,200.0
WARN,1,302.0


## 7. 中文文本归一化

中文语料里最烦的噪声：

- 全角字母数字：`ＡＢＣ１２３`；
- 零宽字符：`\u200b`、`\ufeff`（从网页复制来的常见）；
- 全角空格：`\u3000`；
- 各种换行、制表符、连续空行。

统一做法：`str.normalize("NFKC")` → 去零宽 → 全角空格转半角 → 压缩空白。

In [15]:
cn = pd.Series([
    "ｐａｎｄａｓ　数据处理\u200b",
    "\ufeff开头的 BOM 字符",
    "多行\n\n\n文本\t带 制表符",
])

def normalize_text(s: pd.Series, keep_newline: bool = True) -> pd.Series:
    s = s.astype("string").str.normalize("NFKC")      # 全角 → 半角
    s = s.str.replace("[\u200b\u200c\u200d\ufeff]", "", regex=True)  # 零宽字符（直接写字符，
                                                                     # 不要写成 r"\u200b"：pandas 3 的正则引擎不认 \u 转义）
    s = s.str.replace("\u3000", " ", regex=False)      # 全角空格
    if keep_newline:
        s = s.str.replace(r"[^\S\n]+", " ", regex=True)    # 行内连续空白 → 一个空格
        s = s.str.replace(r"\n{3,}", "\n\n", regex=True)   # 3 个以上换行 → 2 个
    else:
        s = s.str.replace(r"\s+", " ", regex=True)
    return s.str.strip()

print(normalize_text(cn).tolist())

['pandas 数据处理', '开头的 BOM 字符', '多行\n\n文本 带 制表符']


## 8. 词频统计：不需要分词库也能做

英文按 `[a-z]+` 抽词；中文没有空格，最简单的替代是**字符 n-gram**（取相邻 2 个字为一组），
足以看出高频短语。真正的中文分词需要 `jieba` 等工具，但思路完全一样：拆成 token 列表 → 展开成多行 → 计数。

In [16]:
en_text = pd.Series([
    "pandas is a Python library for data analysis",
    "pandas provides DataFrame and Series",
])
tokens = en_text.str.lower().str.findall(r"[a-z]+")
tokens.explode().value_counts().head(8)

pandas      2
is          1
a           1
python      1
library     1
for         1
data        1
analysis    1
Name: count, dtype: int64

In [17]:
# 中文：字符 2-gram
zh = "数据分析 数据清洗 数据可视化 模型训练"
flat = zh.replace(" ", "")
bigrams = [flat[i : i + 2] for i in range(len(flat) - 1)]
pd.Series(bigrams).value_counts().head(6)

数据    3
据分    1
分析    1
析数    1
据清    1
清洗    1
Name: count, dtype: int64

## 9. 小练习

基于 `remarks` 完成：

1. 提取每行的手机号（可能带 `-`，也可能不带）；
2. 把手机号打码成 `138****5678`；
3. 提取金额（`899.00 元` 或 `¥129.9` 两种写法都要能识别）；
4. 给每行加一个 `is_refund` 列，判断是否包含"退款"。

In [18]:
phone = remarks.str.extract(r"(1[3-9]\d(?:-?\d{4}){2})", expand=False)
phone_norm = phone.str.replace("-", "", regex=False)
phone_masked = phone_norm.str.replace(r"(\d{3})\d{4}(\d{4})", r"\1****\2", regex=True)

result = pd.DataFrame({
    "原文": remarks,
    "手机号": phone_norm,
    "打码": phone_masked,
    "金额": remarks.str.extract(r"(?:金额\s*|¥\s*)(\d+\.?\d*)", expand=False),
    "是否退款": remarks.str.contains("退款", na=False),
})
result

,原文,手机号,打码,金额,是否退款
0,订单 A1001 已发货，联系电话 138-1234-5678，金额 899.00 元,13812345678,138****5678,899.00,False
1,客户要求：发票抬头「北京晨光科技有限公司」,NaN,NaN,NaN,False
2,加急！！！请于 2024-03-20 前送达，联系人 王女士 13912345678,13912345678,139****5678,NaN,False
3,退货原因：尺寸不合适，申请退款 ¥129.9,NaN,NaN,129.9,True
4,多余空格 和\n换行 需要清理,NaN,NaN,NaN,False
5,NaN,NaN,NaN,NaN,False


## 10. 小结

**`.str` 方法速查**

| 想做的事 | 方法 |
| --- | --- |
| 长度 / 去空格 | `str.len()` / `str.strip()` |
| 大小写 | `str.lower()` / `str.upper()` |
| 判断包含 / 开头结尾 | `str.contains()` / `str.startswith()` / `str.endswith()` |
| 替换 | `str.replace(pat, rep, regex=True)` |
| 抽取 | `str.extract()` / `str.extractall()` |
| 拆分 | `str.split(sep, expand=True)` / `.explode()` |
| 拼接 | `str.cat(sep=)` / `str.join()` |
| 取字符 / 切片 | `str.get(i)` / `str.slice(a, b)` |
| 补零 / 补齐 | `str.zfill(n)` / `str.pad(n)` |
| 归一化 | `str.normalize("NFKC")` |
| 找全部匹配 | `str.findall()` |

**记住三点**

1. `str.replace` 的正则要显式 `regex=True`；`\u200b` 这类转义在 pandas 3 的正则引擎里不被接受，直接写字符。
2. 变换类方法保留缺失，判断类方法要看 dtype；筛选时统一加 `na=False`。
3. 从文本抽字段 = 写一个带命名组的正则 + `str.extract`，不要写 for 循环。

下一节进入聚合统计：`06_分组聚合与透视.ipynb`。